In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.silver_trades;

In [0]:
UPDATE dbx_batch_mini_ws.batch_mini_project.silver_trades
SET symbol = 'TEST'
WHERE trade_id = 'T0000001';

In [0]:
SELECT trade_id, symbol
FROM dbx_batch_mini_ws.batch_mini_project.silver_trades
WHERE trade_id = 'T0000001';

In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.silver_trades;

In [0]:
UPDATE dbx_batch_mini_ws.batch_mini_project.silver_trades
SET symbol = 'BP'
WHERE trade_id = 'T0000001';

In [0]:
SELECT trade_id, symbol
FROM dbx_batch_mini_ws.batch_mini_project.silver_trades
WHERE trade_id = 'T0000001';

In [0]:
CREATE OR REPLACE TABLE dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test
AS
SELECT trade_id, symbol, quantity, price
FROM dbx_batch_mini_ws.batch_mini_project.silver_trades
LIMIT 5;

In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test;

In [0]:
UPDATE dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test
SET symbol = 'TIME_TRAVEL_TEST'
WHERE trade_id = 'T0000001';

In [0]:
SELECT *
FROM dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test;

In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test;

In [0]:
SELECT *
FROM dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test
VERSION AS OF 0;

In [0]:
SELECT *
FROM dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test;

In [0]:
%python
try:
    spark.sql("""
        INSERT INTO dbx_batch_mini_ws.batch_mini_project.delta_time_travel_test
        (trade_id, symbol, quantity, price)
        VALUES ('SCHEMA_TEST', 'TEST', 'NOT_A_NUMBER', 100.00)
    """)
    print("Unexpected: insert succeeded (schema enforcement did not trigger)")
except Exception as e:
    print(f"Expected failure — schema enforcement rejected the insert:\n{e}")


**Expected result:** this INSERT fails with `CAST_INVALID_INPUT` — this demonstrates Delta Lake's schema enforcement rejecting a malformed value (`'NOT_A_NUMBER'`) for a DECIMAL column. A successful insert here would indicate a schema enforcement problem.


In [0]:
CREATE OR REPLACE TABLE dbx_batch_mini_ws.batch_mini_project.overwrite_append_test (
    id INT,
    name STRING
)
USING DELTA;

In [0]:
INSERT INTO dbx_batch_mini_ws.batch_mini_project.overwrite_append_test
VALUES
(1, 'Alice'),
(2, 'Bob');

In [0]:
SELECT * 
FROM dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;

In [0]:
INSERT INTO dbx_batch_mini_ws.batch_mini_project.overwrite_append_test
VALUES
(3, 'Charlie');

In [0]:
SELECT * 
FROM dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;

In [0]:
INSERT OVERWRITE dbx_batch_mini_ws.batch_mini_project.overwrite_append_test
VALUES
(10, 'David'),
(20, 'Emma');

In [0]:
SELECT * 
FROM dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;

## OPTIMIZE, Compaction and VACUUM

### OPTIMIZE
OPTIMIZE can be used to improve Delta table file layout and reduce the impact of many small files.

### Compaction
Compaction combines small files into fewer larger files. This helps control small-file generation and can improve query performance.

### VACUUM
VACUUM removes old files that are no longer needed by Delta after the retention period.

### Policy
OPTIMIZE and VACUUM should be applied carefully to important data. Time Travel depends on retained Delta history, so old files should not be removed without considering the required retention policy.

In [0]:
DESCRIBE DETAIL dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;


In [0]:
OPTIMIZE dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;


In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;


In [0]:
DESCRIBE DETAIL dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;


In [0]:
VACUUM dbx_batch_mini_ws.batch_mini_project.overwrite_append_test DRY RUN;


In [0]:
VACUUM dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;


In [0]:
DESCRIBE HISTORY dbx_batch_mini_ws.batch_mini_project.overwrite_append_test;
